# 1. Data Cleaning

## Objective

The objective of this step is to prepare the raw cardiac-failure data for reliable analysis.

We will:
- Load and inspect all available datasets
- Understand their structure and data types
- Identify the patient-level key used to connect datasets
- Assess missing values and duplicate records
- Validate categorical and numerical variables using the data dictionary
- Apply only justified cleaning and transformation rules
- Preserve clinically meaningful information wherever possible

The cleaned datasets will serve as the foundation for further analysis.


### 1. Import required libraries

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path


pd.set_option("display.max_columns", None)

### 2. Define the Data Location

In [22]:
DATA_PATH = Path(
    r"C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026"
)

CLEANED_PATH = DATA_PATH / "cleaned_data"
CLEANED_PATH.mkdir(exist_ok=True)

print("Raw data folder:", DATA_PATH)
print("Cleaned data folder:", CLEANED_PATH)

Raw data folder: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026
Cleaned data folder: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data


### 3. Load the Datasets

In [11]:
demography = pd.read_csv(DATA_PATH / "demography.csv")
hospitalization = pd.read_csv(DATA_PATH / "hospitalization_discharge.csv")
cardiac = pd.read_csv(DATA_PATH / "cardiac_complications.csv")
labs = pd.read_csv(DATA_PATH / "labs.csv")
patient_prescriptions = pd.read_csv(DATA_PATH / "patient_precriptions.csv")
patient_history = pd.read_csv(DATA_PATH / "patienthistory.csv")
responsiveness = pd.read_csv(DATA_PATH / "responsivenes.csv")

data_dictionary = pd.read_excel(
    DATA_PATH / "Cardiac_failure_data_dictionary.xlsx"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


### 4. Understand each dataset
Before applying any cleaning rules, we first examine the structure of each dataset.

This helps us understand:
- the number of records and variables
- the relative size of each dataset
- whether datasets appear to contain one record per patient or multiple records per patient
- which datasets may require special handling before integration

This assessment will guide the cleaning and integration decisions that follow.

In [14]:
datasets = {
    "Demography": demography,
    "Hospitalization & Discharge": hospitalization,
    "Cardiac Complications": cardiac,
    "Labs": labs,
    "Patient Prescriptions": patient_prescriptions,
    "Patient History": patient_history,
    "Responsiveness": responsiveness
}

overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Unique Patients": df["inpatient_number"].nunique(),
        "Duplicate Patient IDs": df["inpatient_number"].duplicated().sum(),
        "Patients with Multiple Records": df["inpatient_number"].value_counts().gt(1).sum(),
        "One Row per Patient": df.shape[0] == df["inpatient_number"].nunique(),
    }
    for name, df in datasets.items()
])

overview

,Dataset,Rows,Columns,Unique Patients,Duplicate Patient IDs,Patients with Multiple Records,One Row per Patient
0,Demography,2009,7,2009,0,0,True
1,Hospitalization & Discharge,2008,21,2008,0,0,True
2,Cardiac Complications,2008,14,2008,0,0,True
3,Labs,2008,107,2008,0,0,True
4,Patient Prescriptions,15362,2,2007,13355,1998,False
5,Patient History,2008,17,2008,0,0,True
6,Responsiveness,2008,6,2008,0,0,True


### 5. Patient ID Consistency Check

The datasets were compared using `inpatient_number` to identify differences in patient coverage across sources.

The comparison identified:
- One patient ID present in the demographic dataset but absent from the hospitalization dataset.
- One hospitalization patient with no corresponding prescription record.

These records are retained at this stage because a mismatch or missing prescription record does not, by itself, indicate an invalid patient record.

The affected records will be investigated further before any removal or imputation decision is made.

In [17]:
master_ids = set(hospitalization["inpatient_number"])

for name, df in datasets.items():
    if name == "Patient Prescriptions":
        continue
    
    dataset_ids = set(df["inpatient_number"])
    
    missing_from_master = dataset_ids - master_ids
    missing_from_dataset = master_ids - dataset_ids
    
    if missing_from_master or missing_from_dataset:
        print(f"\n{name}")
        
        if missing_from_master:
            print("Present in dataset but not hospitalization:",
                  missing_from_master)
        
        if missing_from_dataset:
            print("Present in hospitalization but not dataset:",
                  missing_from_dataset)

missing_from_prescriptions = (
    master_ids - set(patient_prescriptions["inpatient_number"])
)

print("\nPatients with no prescription record:",
      missing_from_prescriptions)


Demography
Present in dataset but not hospitalization: {5}

Patients with no prescription record: {789308}


### Excluding Confirmed Invalid Patient Record

Patient ID 5 was identified as a data-entry/placeholder artifact based on its invalid ID format, absence from the other clinical datasets, and extensive missing demographic information.

In [20]:
demography = demography[
    demography["inpatient_number"] != 5
].reset_index(drop=True)

print("demography shape after exclusion:", demography.shape)
print("unique patients:", demography["inpatient_number"].nunique())

demography shape after exclusion: (2008, 7)
unique patients: 2008


#### Update the datasets dictionary
Because our datasets dictionary was created before we removed ID 5, update only the

In [23]:
datasets["Demography"] = demography

print(datasets["Demography"].shape)

(2008, 7)


### 6. Missing-Value Assessment

Missing values are assessed across all datasets before applying any cleaning rules.

The objective is to identify:
- which variables contain missing values
- the number and percentage of missing values
- datasets or variables with substantial missingness

No values are removed or imputed at this stage. Cleaning decisions will be based on the extent and context of the missing data.

### 6A. Detailed Missing-Value Summary

In [18]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        
        if missing_count > 0:
            missing_summary.append({
                "Dataset": name,
                "Column": column,
                "Missing Count": missing_count,
                "Missing %": round((missing_count / len(df)) * 100, 2)
            })

missing_summary = pd.DataFrame(missing_summary)

missing_summary = (
    missing_summary
    .sort_values(["Dataset", "Missing %"], ascending=[True, False])
    .reset_index(drop=True)
)

missing_summary

,Dataset,Column,Missing Count,Missing %
0,Cardiac Complications,tricuspid_valve_return_pressure,1826,90.94
1,Cardiac Complications,ea,1615,80.43
2,Cardiac Complications,mitral_valve_ams,1458,72.61
3,Cardiac Complications,lvef,1373,68.38
4,Cardiac Complications,tricuspid_valve_return_velocity,1218,60.66
...,...,...,...,...
115,Labs,sodium,11,0.55
116,Patient History,cci_score,5,0.25
117,Patient History,peptic_ulcer_disease,2,0.10
118,Patient History,moderate_to_severe_chronic_kidney_disease,2,0.10


### 6B. Missing Values by Dataset

In [19]:
missing_by_dataset = (
    missing_summary
    .groupby("Dataset")
    .agg(
        Columns_with_Missing=("Column", "count"),
        Total_Missing_Values=("Missing Count", "sum")
    )
    .reset_index()
)

missing_by_dataset

,Dataset,Columns_with_Missing,Total_Missing_Values
0,Cardiac Complications,7,9215
1,Demography,5,32
2,Hospitalization & Discharge,5,6149
3,Labs,99,48352
4,Patient History,4,10


### 6C. Variables With Highest Missingness

In [25]:
top_missing = (
    missing_summary
    .sort_values("Missing %", ascending=False)
    .head(20)
)

top_missing

,Dataset,Column,Missing Count,Missing %
17,Labs,cholinesterase,2008,100.00
12,Hospitalization & Discharge,respiratory_support,1966,97.91
13,Hospitalization & Discharge,time_of_death__days_from_admission,1964,97.81
18,Labs,homocysteine,1862,92.73
19,Labs,apolipoprotein_a,1832,91.24
20,Labs,apolipoprotein_b,1832,91.24
21,Labs,lipoprotein,1832,91.24
0,Cardiac Complications,tricuspid_valve_return_pressure,1826,90.94
22,Labs,erythrocyte_sedimentation_rate,1701,84.71
1,Cardiac Complications,ea,1615,80.43


## Conclusion — Phase 1: Cross-Dataset Structural Audit

Before cleaning any individual table, we established the structural facts that govern every cleaning decision made from this point forward:

- **Grain of each table** — 6 datasets are one row per patient; `patient_precriptions` is one-to-many (2,007 patients, 15,362 rows), so it cannot be merged or cleaned using the same logic as the rest.
- **Patient-ID integrity** — every table's population matches the 2,008-patient master set defined by `hospitalization`, with two exceptions: an orphan record (`inpatient_number = 5`) in `demography`, confirmed as a data artifact (wrong ID format, no data in any other table) and excluded; and patient `789308`, who is a valid patient with no prescription record — preserved, to be flagged later rather than treated as "zero medications."
- **Global missingness landscape** (Cells 6A–6C) — a ranked view across all 120 columns with missing data, which tells us *where* the largest gaps are before we look at any single table in isolation. This is kept as a standing reference, not repeated table-by-table.

These checks required comparing tables against each other and could only be done at the dataset level — that's why they came first.

## Phase 2 — Table-by-Table Cleaning

The cross-dataset audit established the overall data-quality landscape. The next phase focuses on cleaning and validating each dataset independently.

A dataset-level approach is used because each source has a different structure, variable meaning, level of measurement, and expected data quality. This allows missing values, data types, duplicates, and invalid values to be evaluated according to the purpose and definition of each variable rather than applying the same rule across all datasets.
The previously completed cross-dataset missingness summary remains as a reference so that decisions remain consistent across datasets.

### Demography — Step 1: Structure, Data Types & Duplicate Check

**Why this step:** Demography is cleaned first because we've already partially investigated it (the ID-5 exclusion) and it's the smallest table — a good one to confirm the table-by-table workflow on before moving to larger datasets like Labs. Before assessing missing values or invalid entries, we confirm each column's data type, take a visual sample of actual records, and check for two distinct duplication risks — identical full rows, and a repeated patient ID — since Demography is expected to be exactly one row per patient.

In [28]:
datasets["Demography"] = demography

print("Demography shape:", demography.shape)
print("\nData types:")
print(demography.dtypes)

print("\nSample records:")
display(demography.head())

print("\nFull-row duplicates:", demography.duplicated().sum())
print("Duplicate patient IDs:", demography["inpatient_number"].duplicated().sum())

Demography shape: (2008, 7)

Data types:
inpatient_number      int64
gender               object
weight              float64
height              float64
bmi                 float64
occupation           object
agecat               object
dtype: object

Sample records:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
0,827040,Female,50.0,1.45,23.781213,NaN,69-79
1,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
2,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
3,866418,Male,70.0,1.70,24.221453,farmer,59-69
4,775928,Male,65.0,1.70,22.491349,UrbanResident,69-79



Full-row duplicates: 0
Duplicate patient IDs: 0


### Demography — Step 2: Missing-Value Detail

**Why this step:** With the ID-5 artifact excluded and duplication ruled out, we look at missing values column-by-column — count, percentage, and data type together — to confirm exactly which fields still need attention before we move to value-level inspection.

In [29]:
demography_missing = pd.DataFrame({
    "Missing Count": demography.isna().sum(),
    "Missing %": (demography.isna().mean() * 100).round(2),
    "Data Type": demography.dtypes.astype(str)
})

demography_missing = (
    demography_missing[demography_missing["Missing Count"] > 0]
    .sort_values("Missing %", ascending=False)
)

demography_missing

,Missing Count,Missing %,Data Type
occupation,27,1.34,object


### Demography — Step 3: Value Inspection

**Why this step:** Before deciding how to treat any column, we look at the actual distribution of every field — including categorical ones — to catch impossible values (e.g., `weight = 0`), implausible ranges (e.g., `height`), and any unexpected category labels. No values are changed at this stage; this is investigation only, consistent with our detect-before-transform rule.

In [30]:
demography.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
inpatient_number,2008.0,NaN,NaN,NaN,797747.542829,41127.80174,722128.0,763164.5,798758.0,829399.75,905720.0
gender,2008,2,Female,1163,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,2008.0,NaN,NaN,NaN,52.483715,10.895935,0.0,45.0,50.0,60.0,115.0
height,2008.0,NaN,NaN,NaN,1.567869,0.096138,0.35,1.5,1.56,1.62,1.83
bmi,2008.0,NaN,NaN,NaN,21.791398,13.645009,0.0,18.491124,20.761246,23.4375,404.081633
occupation,1981,5,UrbanResident,1670,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agecat,2008,8,69-79,715,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Demography — Step 4: Targeted Value Validation

The distribution review identified potentially invalid or implausible values in `weight`, `height`, and `bmi`, while categorical variables require validation of their actual category labels. We now investigate these findings at the record level before deciding whether any values require correction.
No assumptions or transformations are applied until the suspicious records are examined.

In [31]:
print("Weight = 0:")
display(demography[demography["weight"] == 0])

print("\nHeight outside plausible adult range (<1.2m or >2.2m):")
display(demography[(demography["height"] < 1.2) | (demography["height"] > 2.2)])

print("\nBMI = 0:")
display(demography[demography["bmi"] == 0])

print("\nBMI > 100:")
display(demography[demography["bmi"] > 100])

print("\nAll occupation categories:")
print(demography["occupation"].value_counts(dropna=False))

print("\nAll agecat categories:")
print(demography["agecat"].value_counts(dropna=False).sort_index())

Weight = 0:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
379,730511,Female,0.0,1.55,0.0,UrbanResident,79-89
388,785878,Male,0.0,1.65,0.0,UrbanResident,59-69
466,775572,Female,0.0,1.50,0.0,UrbanResident,69-79



Height outside plausible adult range (<1.2m or >2.2m):


,inpatient_number,gender,weight,height,bmi,occupation,agecat
317,837041,Female,49.0,0.48,212.673611,UrbanResident,69-79
320,815731,Male,49.5,0.35,404.081633,farmer,49-59
335,805044,Male,50.0,0.60,138.888889,UrbanResident,59-69
562,844739,Female,49.5,0.35,404.081633,UrbanResident,79-89



BMI = 0:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
379,730511,Female,0.0,1.55,0.0,UrbanResident,79-89
388,785878,Male,0.0,1.65,0.0,UrbanResident,59-69
466,775572,Female,0.0,1.50,0.0,UrbanResident,69-79



BMI > 100:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
317,837041,Female,49.0,0.48,212.673611,UrbanResident,69-79
320,815731,Male,49.5,0.35,404.081633,farmer,49-59
335,805044,Male,50.0,0.60,138.888889,UrbanResident,59-69
562,844739,Female,49.5,0.35,404.081633,UrbanResident,79-89



All occupation categories:
occupation
UrbanResident    1670
farmer            198
Others             89
NaN                27
worker             17
Officer             7
Name: count, dtype: int64

All agecat categories:
agecat
21-29       4
29-39      12
39-49      56
49-59     106
59-69     368
69-79     715
79-89     646
89-110    101
Name: count, dtype: int64


### Demography — Step 5: Validate BMI Consistency

**Why this step:** BMI is mathematically derived from weight and height. Before modifying the identified BMI outliers, we verify whether the BMI values are internally consistent with the recorded weight and height. This helps determine whether the issue is with the BMI calculation itself or with one of its underlying measurements.

In [32]:
demography["calculated_bmi"] = (
    demography["weight"] / demography["height"]**2
)

bmi_check = demography[
    demography["bmi"].notna() &
    demography["calculated_bmi"].notna()
].copy()

bmi_check["bmi_difference"] = (
    bmi_check["bmi"] - bmi_check["calculated_bmi"]
).abs()

display(
    bmi_check.sort_values(
        "bmi_difference",
        ascending=False
    )[[
        "inpatient_number",
        "weight",
        "height",
        "bmi",
        "calculated_bmi",
        "bmi_difference"
    ]].head(10)
)

,inpatient_number,weight,height,bmi,calculated_bmi,bmi_difference
320,815731,49.5,0.35,404.081633,404.081633,2.842171e-13
562,844739,49.5,0.35,404.081633,404.081633,2.842171e-13
317,837041,49.0,0.48,212.673611,212.673611,1.136868e-13
335,805044,50.0,0.60,138.888889,138.888889,1.136868e-13
123,824845,53.0,1.55,22.060354,22.060354,5.329071e-14
377,840155,53.0,1.55,22.060354,22.060354,5.329071e-14
374,742244,57.0,1.35,31.275720,31.275720,5.329071e-14
1817,763848,53.0,1.55,22.060354,22.060354,5.329071e-14
1721,737932,40.0,1.45,19.024970,19.024970,4.973799e-14
921,779958,40.0,1.45,19.024970,19.024970,4.973799e-14


### Demography — Step 6: Data Dictionary Validation

**Why this step:** The BMI consistency check confirmed that the extreme BMI values are mathematically consistent with the recorded weight and height. We therefore investigate the variable definitions in the data dictionary before deciding how to treat the suspicious height and weight values.

In [34]:
print(data_dictionary.shape)
print(data_dictionary.columns.tolist())
display(data_dictionary.head(10))

(166, 3)
['Unnamed: 0', 'Variables', 'Description']


,Unnamed: 0,Variables,Description
0,1,inpatient_number,The patient unique ID
1,2,DestinationDischarge,"Destination of hospital discharge, recorded af..."
2,3,admission_ward,first admission ward
3,4,admission_way,possible ways of admission are Emergency vs. n...
4,5,occupation,occupation
5,6,discharge_department,the patient discharged from
6,7,visit_times,the number of hospital admissions before this ...
7,8,gender,"gender: Male, Female"
8,9,body_temperature,body temperature in degrees celsius
9,10,pulse,pulse rate (beats per minute)


### Demography — Step 7: Cleaning Invalid Measurements
**Finding:** Weight = 0 for 3 patients and height outside a documented adult-plausibility range (1.2–2.2m, per the data dictionary confirming height is measured in meters) for 4 patients. The BMI-consistency check confirmed BMI is mathematically derived from weight/height with no independent calculation error — so BMI is only ever wrong as a *consequence* of an invalid weight or height, never on its own.

**Cleaning decision:** Invalid weight and height values are treated as missing, not deleted — the patient records themselves are valid. BMI is recomputed only where both weight and height are valid, rather than trusting or individually nulling the original BMI column, since we've proven the two approaches are equivalent here but recomputation is self-correcting if that weren't the case.

In [39]:
HEIGHT_MIN_M = 1.2

# Identify invalid measurements
invalid_weight = demography["weight"] <= 0
invalid_height = demography["height"] < HEIGHT_MIN_M

# Treat invalid measurements as missing
demography.loc[invalid_weight, "weight"] = np.nan
demography.loc[invalid_height, "height"] = np.nan

# Recalculate BMI using valid weight and height
valid_bmi_inputs = (
    demography["weight"].notna() &
    demography["height"].notna()
)

demography.loc[valid_bmi_inputs, "bmi"] = (
    demography.loc[valid_bmi_inputs, "weight"]
    / demography.loc[valid_bmi_inputs, "height"] ** 2
)

# BMI cannot be calculated when either input is missing
demography.loc[~valid_bmi_inputs, "bmi"] = np.nan

print(
    "Invalid weights treated as missing:", invalid_weight.sum(),
    "| Invalid heights treated as missing:", invalid_height.sum()
)


Invalid weights treated as missing: 0 | Invalid heights treated as missing: 0


### Demography — Step 8: Standardize Categorical Columns

**Why this step:** `gender`, `occupation`, and `agecat` were confirmed clean (no stray casing/whitespace variants) during value inspection — no values need correction. This step only casts them to `category` dtype for consistency with the rest of the project, and renames `agecat` to `age_category` to match the underlying-word naming convention used elsewhere (`admission_ward`, `discharge_department`, etc.).

In [41]:
demography["gender"] = demography["gender"].astype("category")
demography["occupation"] = demography["occupation"].astype("category")

demography = demography.rename(columns={"agecat": "age_category"})
demography["age_category"] = demography["age_category"].astype("category")

datasets["Demography"] = demography
print(demography.dtypes)

inpatient_number       int64
gender              category
weight               float64
height               float64
bmi                  float64
occupation          category
age_category        category
dtype: object


### Demography — Step 9: Data Type Validation — No Transformation Required

**Finding:** The categorical variables `gender`, `occupation`, and `age_category` are already stored as the `category` data type. The numeric variables also have appropriate numeric types for analysis.

**Decision:** No data type conversion is required. The existing data types are retained.

### Demography — Step 9: Occupation Category Standardization

**Why this step:** The occupation variable contains inconsistent capitalization and formatting across category labels, such as `UrbanResident`, `farmer`, `worker`, and `Officer`. Since these labels represent categorical values, inconsistent formatting can create unnecessary separate categories during grouping and analysis.

The labels are standardized for consistent presentation and aggregation without changing the underlying occupation category.

In [43]:
demography["occupation"].value_counts(dropna=False)

occupation
Urban Resident    1670
Farmer             198
Others              89
NaN                 27
Worker              17
Officer              7
Name: count, dtype: int64

In [42]:
occupation_mapping = {
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "Others": "Others",
    "worker": "Worker",
    "Officer": "Officer"
}

demography["occupation"] = (
    demography["occupation"]
    .map(occupation_mapping)
    .astype("category")
)

### Demography — Step 10: BMI Precision Standardization

**Why this step:** BMI is a derived continuous measurement and currently contains several decimal places due to mathematical calculation. Rounding BMI to two decimal places improves readability and avoids implying precision beyond what is meaningful for the measurement, while retaining sufficient precision for downstream analysis.

In [44]:
demography["bmi"] = demography["bmi"].round(2)

### Demography — Step 11: Post-Cleaning Validation

**Why this step:** After applying the cleaning and standardization rules, we validate the final dataset to confirm that the intended issues were resolved without changing the patient population or introducing new inconsistencies.

The validation checks patient-level uniqueness, remaining missing values, measurement validity, BMI consistency, categorical values, and final dataset structure before export.

In [45]:
print("Final shape:", demography.shape)
print("Unique patients:", demography["inpatient_number"].nunique())
print("Duplicate patient IDs:", demography["inpatient_number"].duplicated().sum())
print("Full-row duplicates:", demography.duplicated().sum())

print("\nInvalid weight values:")
print((demography["weight"] <= 0).sum())

print("\nInvalid height values:")
print((demography["height"] < 1.2).sum())

print("\nBMI outside 0–100:")
print(((demography["bmi"] <= 0) | (demography["bmi"] > 100)).sum())

print("\nMissing values:")
display(demography.isna().sum())

print("\nOccupation categories:")
print(demography["occupation"].value_counts(dropna=False))

print("\nData types:")
print(demography.dtypes)

Final shape: (2008, 7)
Unique patients: 2008
Duplicate patient IDs: 0
Full-row duplicates: 0

Invalid weight values:
0

Invalid height values:
0

BMI outside 0–100:
0

Missing values:


inpatient_number     0
gender               0
weight               3
height               4
bmi                  7
occupation          27
age_category         0
dtype: int64


Occupation categories:
occupation
Urban Resident    1670
Farmer             198
Others              89
NaN                 27
Worker              17
Officer              7
Name: count, dtype: int64

Data types:
inpatient_number       int64
gender              category
weight               float64
height               float64
bmi                  float64
occupation          category
age_category        category
dtype: object


In [46]:
valid_bmi = (
    demography["weight"].notna() &
    demography["height"].notna() &
    demography["bmi"].notna()
)

calculated_bmi = (
    demography.loc[valid_bmi, "weight"]
    / demography.loc[valid_bmi, "height"] ** 2
)

bmi_difference = (
    demography.loc[valid_bmi, "bmi"] - calculated_bmi
).abs()

print("Maximum BMI rounding difference:", bmi_difference.max())

Maximum BMI rounding difference: 0.004999999999997229


In [47]:
(demography["bmi"] <= 0) | (demography["bmi"] > 100)

0       False
1       False
2       False
3       False
4       False
        ...  
2003    False
2004    False
2005    False
2006    False
2007    False
Name: bmi, Length: 2008, dtype: bool

### Demography — Validation Conclusion

The Demography dataset passed all post-cleaning validation checks.

The dataset contains 2,008 unique patients with no duplicate patient IDs or duplicate rows. Invalid weight and height values identified during validation were treated as missing, and BMI was recalculated only where valid measurements were available.

Occupation labels were standardized, column naming was improved, and BMI was rounded to two decimal places. The final dataset contains no remaining invalid weight, height, or BMI values.

The cleaned Demography dataset is therefore ready for export and downstream analysis.

In [48]:
demography.to_csv(
    CLEANED_PATH / "demography_cleaned.csv",
    index=False
)

print("Saved:", CLEANED_PATH / "demography_cleaned.csv")

Saved: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\demography_cleaned.csv


In [49]:
demography_check = pd.read_csv(
    CLEANED_PATH / "demography_cleaned.csv"
)

print("Saved shape:", demography_check.shape)
print("Saved columns:", demography_check.columns.tolist())

Saved shape: (2008, 7)
Saved columns: ['inpatient_number', 'gender', 'weight', 'height', 'bmi', 'occupation', 'age_category']
